# E-Commerce Business-Focused Exploratory Data Analysis

This notebook performs a complete exploratory data analysis (EDA) for e-commerce datasets. The analysis is structured to support business insights, operational priorities, and next-stage analytics planning.

Key objectives:
- Understand dataset scope, volume, and schema.
- Verify data cleaning results and missing values.
- Generate descriptive statistics and examine feature distributions.
- Analyze categorical business segments, geography, and product performance.
- Detect outliers, assess impact, and identify data or business anomalies.
- Explore relationships among revenue, quantity, and operational variables.
- Summarize actionable findings and recommend next-stage analyses.

In [1]:
# Standard libraries for data analysis and visualization
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("data/cleaned")


def load_dataset(file_path: Path) -> pd.DataFrame:
    """Load a cleaned CSV dataset into a pandas DataFrame."""
    return pd.read_csv(file_path)


def summarize_columns(df: pd.DataFrame) -> None:
    """Print a concise summary of dataset schema and data types."""
    display_df = pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing_count": df.isna().sum(),
            "unique_values": df.nunique(dropna=True),
        }
    )
    print(display_df)


def plot_histogram(df: pd.DataFrame, column: str, title: str) -> None:
    """Plot a histogram for a numeric column with meaningful business labels."""
    fig = px.histogram(
        df,
        x=column,
        nbins=40,
        marginal="box",
        title=title,
        labels={column: column},
    )
    fig.update_layout(template="plotly_white")
    fig.show()


def plot_bar_from_counts(counts: pd.Series, title: str, x_label: str, y_label: str) -> None:
    """Plot a bar chart from a value counts series."""
    fig = px.bar(
        x=counts.index.astype(str),
        y=counts.values,
        title=title,
        labels={"x": x_label, "y": y_label},
    )
    fig.update_layout(xaxis_tickangle=-35, template="plotly_white")
    fig.show()


def calculate_iqr_outliers(series: pd.Series) -> tuple[float, float, pd.Series]:
    """Return lower/upper bounds and a boolean mask of outliers using IQR."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    mask = (series < lower_bound) | (series > upper_bound)
    return lower_bound, upper_bound, mask


def business_summary(value: pd.Series, label: str) -> None:
    """Print business-focused summary statistics for a numeric series."""
    print(f"{label}")
    print(f"- Mean: {value.mean():,.2f}")
    print(f"- Median: {value.median():,.2f}")
    print(f"- Minimum: {value.min():,.2f}")
    print(f"- Maximum: {value.max():,.2f}")
    print(f"- Standard deviation: {value.std():,.2f}")
    print(f"- Missing values: {value.isna().sum():,}")
    print()

## 1. Overview of the Dataset

This section describes the dataset context, business purpose, and key performance indicators (KPIs). The primary goal is to identify revenue drivers, product performance, and operational issues in the e-commerce business.

**Business context:**
- The dataset includes sales and product metadata from an e-commerce operation.
- Key areas of interest are order performance, product categories, fulfillment channels, and geographic distribution.
- The analysis should support decisions about pricing, inventory, customer targeting, and channel optimization.

**Target KPIs:**
- Total sales value and average order value.
- Quantity sold and revenue per product category.
- Contribution by fulfillment type and sales channel.
- Data quality metrics such as missing values, duplicates, and outlier counts.

In [2]:
# Load the primary e-commerce dataset for analysis
primary_file = DATA_DIR / "amazon_sale_report_cleaned.csv"
primary_df = load_dataset(primary_file)

print(f"Primary dataset loaded from: {primary_file}")
print(f"Rows: {primary_df.shape[0]:,}")
print(f"Columns: {primary_df.shape[1]}")
primary_df.head(5)

Primary dataset loaded from: data/cleaned/amazon_sale_report_cleaned.csv
Rows: 128,969
Columns: 22


,order_id,date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,asin,courier_status,qty,currency,amount,ship_city,ship_state,ship_postal_code,ship_country,promotion_ids,b2b,fulfilled_by
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,NaN,0,INR,647.62,MUMBAI,MAHARASHTRA,"400,081.00",IN,NaN,False,Easy Ship
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,KARNATAKA,"560,085.00",IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,"410,210.00",IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,NaN,0,INR,753.33,PUDUCHERRY,PUDUCHERRY,"605,008.00",IN,NaN,False,Easy Ship
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,INR,574.00,CHENNAI,TAMIL NADU,"600,073.00",IN,NaN,False,NaN


## 2. Data Loading and Initial Inspection

We confirm that the cleaned dataset is ready for EDA by checking the structure and reviewing sample rows. This stage ensures target columns are available for business-focused analysis.

In [3]:
# Inspect the dataset schema and data types
summarize_columns(primary_df)

print("\nSample rows from the dataset:")
primary_df.head(8)

                      dtype  missing_count  unique_values
order_id                str              0         120378
date                    str              0             91
status                  str              0             13
fulfilment              str              0              2
sales_channel           str              0              2
ship_service_level      str              0              2
style                   str              0           1377
sku                     str              0           7195
category                str              0              9
size                    str              0             11
asin                    str              0           7190
courier_status          str           6872              3
qty                   int64              0             10
currency                str           7792              1
amount              float64           7792           1410
ship_city               str             33           8951
ship_state    

,order_id,date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,asin,courier_status,qty,currency,amount,ship_city,ship_state,ship_postal_code,ship_country,promotion_ids,b2b,fulfilled_by
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,NaN,0,INR,647.62,MUMBAI,MAHARASHTRA,"400,081.00",IN,NaN,False,Easy Ship
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,KARNATAKA,"560,085.00",IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,"410,210.00",IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,NaN,0,INR,753.33,PUDUCHERRY,PUDUCHERRY,"605,008.00",IN,NaN,False,Easy Ship
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,INR,574.00,CHENNAI,TAMIL NADU,"600,073.00",IN,NaN,False,NaN
5,404-1490984-4578765,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,SET264,SET264-KR-NP-XL,Set,XL,B08YN7XDSG,Shipped,1,INR,824.00,GHAZIABAD,UTTAR PRADESH,"201,102.00",IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN
6,408-5748499-6859555,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,J0095,J0095-SET-L,Set,L,B08CMHNWBN,Shipped,1,INR,653.00,CHANDIGARH,CHANDIGARH,"160,036.00",IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN
7,406-7807733-3785945,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3405,JNE3405-KR-S,kurta,S,B081WX4G4Q,Shipped,1,INR,399.00,HYDERABAD,TELANGANA,"500,032.00",IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship


## 3. Verify Data Cleaning and Missing Values

In this section we verify that cleaning was applied correctly and highlight any remaining issues. We also validate duplicates and data consistency for key business fields.

In [4]:
# Verify missing values and duplicates
missing_counts = primary_df.isna().sum().sort_values(ascending=False)
missing_counts[missing_counts > 0]

fulfilled_by        89692
promotion_ids       49150
currency             7792
amount               7792
courier_status       6872
ship_country           33
ship_postal_code       33
ship_state             33
ship_city              33
dtype: int64

In [5]:
# Duplicate rows and key consistency checks
duplicate_count = primary_df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

key_fields = ["order_id", "date", "status", "sales_channel", "amount"]
print("\nMissing values in key fields:")
print(primary_df[key_fields].isna().sum())

Duplicate rows: 0

Missing values in key fields:
order_id            0
date                0
status              0
sales_channel       0
amount           7792
dtype: int64


### Business interpretation
- Confirm whether any important business fields still have missing or inconsistent data.
- Missing values in amount, quantity, or order status can distort revenue and fulfillment analysis.
- Duplicates should be verified to prevent double-counting sales.

In [6]:
# Review the most frequent categories and any strange values in selected fields
for field in ["status", "fulfilment", "sales_channel", "courier_status", "category"]:
    print(f"\nTop values for {field}:")
    print(primary_df[field].value_counts(dropna=False).head(10))


Top values for status:
status
Shipped                          77801
Shipped - Delivered to Buyer     28769
Cancelled                        18329
Shipped - Returned to Seller      1953
Shipped - Picked Up                973
Pending                            658
Pending - Waiting for Pick Up      281
Shipped - Returning to Seller      145
Shipped - Out for Delivery          35
Shipped - Rejected by Buyer         11
Name: count, dtype: int64

Top values for fulfilment:
fulfilment
Amazon      89692
Merchant    39277
Name: count, dtype: int64

Top values for sales_channel:
sales_channel
Amazon.in     128845
Non-Amazon       124
Name: count, dtype: int64

Top values for courier_status:
courier_status
Shipped      109484
NaN            6872
Unshipped      6681
Cancelled      5932
Name: count, dtype: int64

Top values for category:
category
Set              50281
kurta            49874
Western Dress    15500
Top              10622
Ethnic Dress      1159
Blouse             926
Bottom       

## 4. Descriptive Statistics and Summary Metrics

This section summarizes numeric fields and key categorical features. Business metrics such as average order value, quantity distribution, and segment counts are useful for stakeholder interpretation.

In [7]:
# Descriptive statistics for numeric fields
numeric_columns = primary_df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", numeric_columns)
primary_df[numeric_columns].describe().T

Numeric columns: ['qty', 'amount', 'ship_postal_code']


,count,mean,std,min,25%,50%,75%,max
qty,"128,969.00",0.90,0.31,0.00,1.00,1.00,1.00,15.00
amount,"121,177.00",648.56,281.21,0.00,449.00,605.00,788.00,"5,584.00"
ship_postal_code,"128,936.00","463,966.66","191,475.02","110,001.00","382,421.00","500,033.00","600,024.00","989,898.00"


In [8]:
# Business summary for key numeric variables
business_summary(primary_df["amount"], "Sales amount summary")
business_summary(primary_df["qty"], "Quantity summary")

Sales amount summary
- Mean: 648.56
- Median: 605.00
- Minimum: 0.00
- Maximum: 5,584.00
- Standard deviation: 281.21
- Missing values: 7,792

Quantity summary
- Mean: 0.90
- Median: 1.00
- Minimum: 0.00
- Maximum: 15.00
- Standard deviation: 0.31
- Missing values: 0



In [9]:
# Summary counts for key categorical fields
categorical_fields = ["status", "fulfilment", "sales_channel", "category", "ship_country"]
for field in categorical_fields:
    unique_count = primary_df[field].nunique(dropna=True)
    print(f"{field}: {unique_count:,} unique values")

for field in categorical_fields:
    print(f"\nTop values for {field}:")
    display(primary_df[field].value_counts(dropna=False).head(10))

status: 13 unique values
fulfilment: 2 unique values
sales_channel: 2 unique values
category: 9 unique values
ship_country: 1 unique values

Top values for status:


status
Shipped                          77801
Shipped - Delivered to Buyer     28769
Cancelled                        18329
Shipped - Returned to Seller      1953
Shipped - Picked Up                973
Pending                            658
Pending - Waiting for Pick Up      281
Shipped - Returning to Seller      145
Shipped - Out for Delivery          35
Shipped - Rejected by Buyer         11
Name: count, dtype: int64


Top values for fulfilment:


fulfilment
Amazon      89692
Merchant    39277
Name: count, dtype: int64


Top values for sales_channel:


sales_channel
Amazon.in     128845
Non-Amazon       124
Name: count, dtype: int64


Top values for category:


category
Set              50281
kurta            49874
Western Dress    15500
Top              10622
Ethnic Dress      1159
Blouse             926
Bottom             440
Saree              164
Dupatta              3
Name: count, dtype: int64


Top values for ship_country:


ship_country
IN     128936
NaN        33
Name: count, dtype: int64

## 5. Distribution Analysis for Numerical Variables

We analyze the distribution of revenue, order quantity, and amount to identify where the business is concentrated. This helps identify whether the data is skewed toward low-value or high-value transactions.

In [10]:
plot_histogram(primary_df, "amount", "Distribution of Sales Amount")
plot_histogram(primary_df, "qty", "Distribution of Order Quantity")

### Business interpretation
- A right-skewed sales amount distribution suggests a long tail of high-value orders.
- Quantity distribution reveals if the business is dominated by single-item purchases or bulk orders.
- These patterns are important for pricing strategy and inventory planning.

## 6. Categorical Variable Analysis and Business Segments

This section evaluates product performance, fulfillment segments, and geographic distribution. We focus on features that directly influence revenue and operational execution.

In [11]:
for field in ["category", "fulfilment", "sales_channel", "courier_status", "ship_country"]:
    counts = primary_df[field].value_counts(dropna=False).head(15)
    plot_bar_from_counts(
        counts,
        title=f"Top {field.replace('_', ' ').title()} Values",
        x_label=field,
        y_label="Count",
    )

category_revenue = primary_df.groupby("category")["amount"].sum().sort_values(ascending=False).head(15)
plot_bar_from_counts(
    category_revenue,
    title="Top Categories by Total Sales Amount",
    x_label="Category",
    y_label="Total Sales Amount",
)

fulfilment_revenue = primary_df.groupby("fulfilment")["amount"].sum().sort_values(ascending=False)
plot_bar_from_counts(
    fulfilment_revenue,
    title="Sales Amount by Fulfilment Type",
    x_label="Fulfilment",
    y_label="Total Sales Amount",
)

### Business interpretation
- High-contribution categories indicate product lines that should be prioritized for inventory and promotion.
- Fulfilment type revenue reveals whether merchant fulfillment or marketplace fulfillment drives top-line sales.
- Geography analysis helps identify markets with the highest order volume and revenue potential.

## 7. Outlier Detection and Impact Assessment

Using the IQR method and boxplots, we identify extreme order values and quantity entries. Business insights from these outliers may indicate premium sales or data issues.

In [12]:
for field in ["amount", "qty"]:
    col_data = primary_df[field].dropna()
    lower, upper, outlier_mask = calculate_iqr_outliers(col_data)
    outlier_count = int(outlier_mask.sum())
    print(
        f"{field.title()} outliers: {outlier_count:,} values outside [{lower:,.2f}, {upper:,.2f}]"
    )
    fig = px.box(
        primary_df,
        y=field,
        title=f"Boxplot for {field.title()}",
        points="outliers",
    )
    fig.update_layout(template="plotly_white")
    fig.show()

    if outlier_count > 0:
        sample = primary_df.loc[outlier_mask.values, ["order_id", "date", "status", field, "category", "sales_channel"]].head(10)
        print(f"Sample outlier rows for {field}:")
        display(sample)

Amount outliers: 3,600 values outside [-59.50, 1,296.50]


IndexError: Boolean index has wrong length: 121177 instead of 128969

### Business interpretation
- Outliers in amount can signal high-value invoices or mispriced transactions.
- Large quantity orders may represent bulk retail or channel partner shipments.
- If outliers are due to data errors, they should be corrected before forecasting or profitability analysis.

## 8. Correlation and Variable Relationships

We explore how revenue correlates with quantity and fulfillment options. This helps surface the operational drivers behind order value and business efficiency.

In [ ]:
correlation = primary_df[["amount", "qty"]].corr()
correlation

In [ ]:
fig = px.imshow(
    correlation,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    title="Correlation Matrix for Amount and Quantity",
)
fig.update_layout(template="plotly_white")
fig.show()

scatter_fig = px.scatter(
    primary_df,
    x="qty",
    y="amount",
    color="category",
    title="Order Amount vs Quantity by Category",
    hover_data=["order_id", "status", "sales_channel"],
)
scatter_fig.update_layout(template="plotly_white")
scatter_fig.show()

### Business interpretation
- A strong positive correlation between amount and quantity suggests larger orders usually generate more revenue.
- Scatter plots can reveal categories with high average order values or exceptional unit economics.
- This relationship informs pricing and bundle strategies.

## 9. Time-based and Customer Behavior Analysis

Time-based analysis is critical for understanding seasonality, order cadence, and customer behavior. We derive a date field and explore sales trends over time.

In [ ]:
# Convert date columns and analyze time trends
primary_df["date"] = pd.to_datetime(primary_df["date"], errors="coerce")
print("Invalid date rows:", primary_df["date"].isna().sum())

sales_by_month = (
    primary_df.dropna(subset=["date"])
    .groupby(primary_df["date"].dt.to_period("M"))["amount"]
    .sum()
    .reset_index()
)
sales_by_month["date"] = sales_by_month["date"].dt.to_timestamp()

fig = px.line(
    sales_by_month,
    x="date",
    y="amount",
    title="Monthly Sales Trend",
    labels={"amount": "Total Sales Amount", "date": "Month"},
)
fig.update_layout(template="plotly_white")
fig.show()

repeat_behavior = (
    primary_df.groupby("ship_city")["order_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(15)
)
plot_bar_from_counts(
    repeat_behavior,
    title="Top Ship Cities by Unique Order Count",
    x_label="Ship City",
    y_label="Unique Orders",
)

### Business interpretation
- Monthly trend analysis helps identify seasonality, promotional uplift, and inventory planning windows.
- City-level order patterns are useful for logistics, regional marketing, and distribution planning.
- A deeper customer-level lifecycle analysis should follow when customer identifiers are available.

## 10. Business Insights from Visualizations

This section translates analytic findings into business recommendations and operational questions.

- The dataset is large enough to provide granular insights across product categories and fulfillment methods.
- Category-level revenue concentration suggests prioritizing inventory and promotions for top-performing product lines.
- Fulfilment type analysis can identify whether marketplace or merchant-fulfilled orders deliver more consistent margin and service levels.
- Outliers in amount and quantity should be interpreted as either premium large orders or potential data anomalies.
- Time-based sales trends highlight the need for better forecasting and promotional planning around peak months.
- Geographic order volume shows where logistics and customer acquisition can be scaled.

## 11. Key Findings and Recommended Next-stage Analyses

### Key findings
- The primary dataset contains over 128,000 orders with strong revenue signals in `amount` and quantity.
- Sales are concentrated in a few high-value categories and fulfillment paths.
- Missing and duplicate checks should be addressed before creating formal reports.
- The amount vs quantity relationship is positive, supporting volume-based pricing and bundling analysis.
- Monthly trends and city-level order concentration point to actionable operational improvements.

### Recommended next-stage analyses
- Customer segmentation and lifetime value modeling once customer identifiers are available.
- Profitability analysis by product category, fulfillment method, and shipping geography.
- Predictive demand forecasting for top categories and peak months.
- Churn and return risk analysis using order status, courier status, and fulfillment performance.
- Cross-channel performance comparison to determine the most profitable sales channels.